In [ ]:
import duckdb

conn = duckdb.connect()

# Obtendo ID dos artigos

In [ ]:
import requests

base_url = base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=pmc&term=(leiomyoma+OR+leiomyomas+OR+fibroids+OR+fibroid+OR+fibromyoma+OR+fibromyomas+OR+uterine+myoma+OR+uterine+myomas)+AND+open+access%5Bfilter%5D&retmax=100000&retmode=json"


response = requests.get(base_url)

if response.status_code == 200:
    data = response.json()
    ids = data['esearchresult']['idlist']
else:
    print(f"Error: {response.status_code}")


In [ ]:
len(ids)

# Obtendo caminho no servidor SFTP

In [ ]:
a_ids = ["PMC" + str(id)  for id in ids] 

In [ ]:
a_ids[:5]

O arquivo oa_file_list.csv pode ser obtido em https://ftp.ncbi.nlm.nih.gov/pub/pmc/ . 

In [ ]:
conn.execute("""
        CREATE TABLE leiomyoma AS
      SELECT * FROM read_csv('./oa_file_list.csv', sample_size = -1) WHERE "Accession ID" IN ? 
""", (a_ids,))

In [ ]:
conn.sql("""
        SELECT * FROM leiomyoma
         """).write_csv("./leiomyoma_articles.csv")

# Obtendo artigos do servidor FTP

In [ ]:
files = conn.sql(
    """
        SELECT File, "Accession ID" FROM "./leiomyoma_articles.csv" 
    """
).fetchall()

In [ ]:
files[:5]

In [ ]:

import urllib.request

base_url = "https://ftp.ncbi.nlm.nih.gov/pub/pmc/"

def download_file(files):
    for file, aid in files:
        urllib.request.urlretrieve(f"{base_url}/{file}", f"/run/media/victor/pessoal/mestrado/codigo/datasets/leiomyoma_files/tar/{aid}.tar.gz")

In [ ]:
def chunkify(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i + n]

In [ ]:
chunks = list(chunkify(files, n = 2000))

In [ ]:
len(chunks)

In [ ]:

import threading

threads = []

for chunk in chunks:
    thread = threading.Thread(target=download_file, args=[chunk])
    thread.start()
    threads.append(thread)
    
for thread in threads:
    thread.join()

# Extrair os arquivos

In [ ]:
import os

tar_files = os.listdir("/run/media/victor/pessoal/mestrado/codigo/datasets/leiomyoma_files/tar")

In [ ]:
len(tar_files)

In [ ]:
tar_files_chunks = list(chunkify(tar_files, n = 1000))

In [ ]:
import tarfile 
import os

def extract_file(tar_files):
    for tar_file in tar_files:
        tar_file = f"/run/media/victor/pessoal/mestrado/codigo/datasets/leiomyoma_files/tar/{tar_file}"
        file = tarfile.open(tar_file) 
        file.extractall("/run/media/victor/pessoal/mestrado/codigo/datasets/leiomyoma_files/articles")
        os.remove(tar_file)    


In [ ]:
threads = []

for chunk in tar_files_chunks:
    thread = threading.Thread(target=extract_file, args=[chunk])
    thread.start()
    threads.append(thread)
    
for thread in threads:
    thread.join()

In [ ]:
tar_files

In [ ]:
import tarfile 
from tqdm import tqdm
import os

with tqdm(total=len(tar_files)) as pbar:
    for tar_file in tar_files:
        tar_file = f"/run/media/victor/pessoal/mestrado/codigo/datasets/leiomyoma_files/tar/{tar_file}"
        file = tarfile.open(tar_file) 
        file.extractall("/run/media/victor/pessoal/mestrado/codigo/datasets/leiomyoma_files/articles")
        os.remove(tar_file)
        pbar.update(1)